In [1]:
import os
import sys
p = os.path.abspath('../..')
if p not in sys.path:
    sys.path.append(p)
import waveorder as wo
import numpy as np
import tifffile as tiff
import matplotlib.pyplot as plt
from numcodecs import Blosc
import glob
import json
from waveorder.visual import plotVectorField
import zarr
from recOrder.recOrder.compute.QLIPP_compute import reconstruct_QLIPP_3D, initialize_reconstructor

In [2]:
def init_zarr_store(data_path):
    chunk_size = (1, 65, 2048, 2448) # Dimension order PZYX
    compressor=Blosc(cname='zstd', clevel=1, shuffle=Blosc.SHUFFLE)
    
    data_shape = (5, 65, 2048, 2448)
    
    if not data_path.endswith('.zarr'):
        data_path += '.zarr'
    
    if not os.path.exists(data_path):
        os.makedirs(data_path)
        
    zarr_store = zarr.open(data_path, mode='a')
    
    
    datasets = ['Phase3D', 'Retardance', 'Orientation', 'BF']
    groups = ['C1', 'C2']
    
    for group in groups:
        zarr_store.create_group(group)
        for ds in datasets:
            zarr_store[group].zeros(ds, shape=data_shape, chunks=chunk_size, dtype='float32',
                                        compressor=compressor)
            
    return zarr_store

In [3]:
def load_bg(bg_path, height, width):
    bg_paths = glob.glob(bg_path+'*.tif')
    bg_paths.sort()
#     print(bg_paths)
    bg_data = np.zeros([len(bg_paths),height,width])
    
    for i in range(len(bg_paths)):
        bg_data[i,:,:] = tiff.imread(bg_paths[i])

    return bg_data

In [12]:
path = '/gpfs/CompMicro/rawdata/hummingbird/Janie/2021_02_03_40x_04NA_A549/48hr_RSV/Coverslip_2/C1_C2_multichannel_stack_1/'

tiff_paths = glob.glob(path+'*.tif')
tiff_paths.sort()


In [13]:
tiff_paths

['/gpfs/CompMicro/rawdata/hummingbird/Janie/2021_02_03_40x_04NA_A549/48hr_RSV/Coverslip_2/C1_C2_multichannel_stack_1/C1_C2_multichannel_stack_1_MMStack.ome.tif',
 '/gpfs/CompMicro/rawdata/hummingbird/Janie/2021_02_03_40x_04NA_A549/48hr_RSV/Coverslip_2/C1_C2_multichannel_stack_1/C1_C2_multichannel_stack_1_MMStack_1.ome.tif',
 '/gpfs/CompMicro/rawdata/hummingbird/Janie/2021_02_03_40x_04NA_A549/48hr_RSV/Coverslip_2/C1_C2_multichannel_stack_1/C1_C2_multichannel_stack_1_MMStack_2.ome.tif',
 '/gpfs/CompMicro/rawdata/hummingbird/Janie/2021_02_03_40x_04NA_A549/48hr_RSV/Coverslip_2/C1_C2_multichannel_stack_1/C1_C2_multichannel_stack_1_MMStack_3.ome.tif',
 '/gpfs/CompMicro/rawdata/hummingbird/Janie/2021_02_03_40x_04NA_A549/48hr_RSV/Coverslip_2/C1_C2_multichannel_stack_1/C1_C2_multichannel_stack_1_MMStack_4.ome.tif',
 '/gpfs/CompMicro/rawdata/hummingbird/Janie/2021_02_03_40x_04NA_A549/48hr_RSV/Coverslip_2/C1_C2_multichannel_stack_1/C1_C2_multichannel_stack_1_MMStack_5.ome.tif',
 '/gpfs/CompMicro/

In [14]:
positions = {}
with tiff.TiffFile(tiff_paths[0]) as tif:
    for idx, tiffpageseries in enumerate(tif.series):
        if idx <= 4:
            print(idx, tiffpageseries)
            positions[idx] = zarr.open(tiffpageseries.aszarr(), mode='r')

0 TiffPageSeries 0  'C1_C2_mul…_MMStack'  6x65x2048x2448  uint16  CZYX  OME  390 Pages
1 TiffPageSeries 1  'C1_C2_mul…_MMStack'  6x65x2048x2448  uint16  CZYX  OME  390 Pages
2 TiffPageSeries 2  'C1_C2_mul…_MMStack'  6x65x2048x2448  uint16  CZYX  OME  390 Pages
3 TiffPageSeries 3  'C1_C2_mul…_MMStack'  6x65x2048x2448  uint16  CZYX  OME  390 Pages
4 TiffPageSeries 4  'C1_C2_mul…_MMStack'  6x65x2048x2448  uint16  CZYX  OME  390 Pages


In [15]:
## Set Reconstruction Parameters
image_dim = (2048,2448)
wavelength = 525
swing = 0.05
N_channel = 4
NA_obj = 1.2
NA_illu = 0.4
mag = 40
N_slices = 65
z_step = 0.25
pad_z = 0
pixel_size = 3.45
bg_option = 'local_fit'
n_media = 1.33

save_dir = '/gpfs/CompMicro/projects/A549/210203_40x_04_NA_A549/'


reconstructor = initialize_reconstructor(image_dim, wavelength, swing, N_channel, NA_obj, NA_illu, mag, N_slices, z_step, pad_z, 
                                         pixel_size, bg_option = bg_option, n_media = n_media, use_gpu=False, gpu_id = 0)

Initializing Reconstructor...
Finished Initializing Reconstructor (1.45 min)


In [16]:
%%time
bg_path = '/gpfs/CompMicro/rawdata/hummingbird/Janie/2021_02_03_40x_04NA_A549/48hr_RSV/BG/'
bg_data = load_bg(bg_path, 2048, 2448)

CPU times: user 31.2 ms, sys: 60.6 ms, total: 91.8 ms
Wall time: 2.02 s


In [17]:
%%time
# zarr_store = init_zarr_store(save_dir+'48hr_RSV')
    
for pos in range(len(positions)):

    ret_stack, ori_stack, BF_stack, phase3D = reconstruct_QLIPP_3D(positions[pos],
                                                           bg_data, reconstructor, method = "Tikhonov",
                                                           reg_re = 1e-4, reg_im = 1e-4, rho = 1e-5, 
                                                           lambda_re = 1e-3, lambda_im = 1e-3, itr = 20)


    zarr_store['C2']['BF'][pos] = BF_stack
    zarr_store['C2']['Retardance'][pos] = ret_stack
    zarr_store['C2']['Orientation'][pos] = ori_stack
    zarr_store['C2']['Phase3D'][pos] = np.transpose(phase3D,(2,0,1))

Computing Birefringence...
Finished Computing Birefringence (1.88 min)
Computing 3d Phase...
Finished Reconstruction (2.49 min)

Computing Birefringence...
Finished Computing Birefringence (1.88 min)
Computing 3d Phase...
Finished Reconstruction (2.40 min)

Computing Birefringence...
Finished Computing Birefringence (1.88 min)
Computing 3d Phase...
Finished Reconstruction (2.40 min)

Computing Birefringence...
Finished Computing Birefringence (1.88 min)
Computing 3d Phase...
Finished Reconstruction (2.41 min)

Computing Birefringence...
Finished Computing Birefringence (1.88 min)
Computing 3d Phase...
Finished Reconstruction (2.40 min)

CPU times: user 12min 34s, sys: 8min 21s, total: 20min 55s
Wall time: 15min 51s


In [ ]:
wo.visual.image_stack_viewer_fast(retardance_stack, vrange=(0,20))

In [ ]:
wo.visual.image_stack_viewer_fast(np.transpose(phase3d,(2,1,0)))

In [ ]:
plt.figure(dpi=300)
plt.imshow(phase3d[:,:,30], 'gray')